# NPClassifierPipeline — unified fit / predict / explain

`NPClassifierPipeline` is a single object that owns the entire NP-Classifier workflow:

| Method | What it does |
|--------|--------------|
| `.fit()` | Train all three hierarchical models (pathway / superclass / class) |
| `.predict()` | Full ontology-based voted prediction |
| `.predict_pathway()` | Pathway-level labels only |
| `.predict_superclass()` | Superclass-level labels only |
| `.predict_class()` | Class-level labels only |
| `.predict_embeddings()` | Penultimate-layer embeddings from a chosen model |
| `.explain()` | Combined three-level SHAP figure |
| `.explain_bits()` | Morgan-fragment grid for one hierarchy level |

**Prerequisites**  
Run `01_training.ipynb` first to produce hierarchical checkpoints, **or** skip section 1 and jump straight to section 2 (pretrained models are downloaded automatically).

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch_np_classifier import NPClassifierPipeline

: 

## Configuration

Adjust paths to match your local setup.

In [ ]:
TRAIN_CSV = "../examples/train_dataset.csv"
TEST_CSV = "../examples/test_dataset.csv"
SMILES_COL = "SMILES"
LABEL_START = 2  # columns to skip: 'key' and 'SMILES'

# Checkpoints written during .fit() (or by 01_training.ipynb)
CKPT_DIR = "pipeline/"
PATHWAY_CKPT = "hierarchical_lifecycle/pathway_np_classifier.ckpt"
SUPERCLASS_CKPT = "hierarchical_lifecycle/superclass_np_classifier.ckpt"
CLASS_CKPT = "hierarchical_lifecycle/class_np_classifier.ckpt"

# Test molecules
MOLECULES = {
    "caffeine": "Cn1cnc2c1c(=O)n(c(=O)n2C)C",
    "quercetin": "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12",
    "morphine": "CN1CC[C@]23c4c5ccc(O)c4O[C@H]2[C@@H](O)C=C[C@@H]3[C@@H]1C5",
    "cholesterol": "[C@@H]1(CC[C@@H]2[C@@]1(CC[C@H]3[C@H]2CC=C4[C@@]3(CC[C@@H](C4)O)C)[C@H](CCCC(C)C)C)C",
    "sucrose": "OC[C@H]1O[C@@](CO)(O[C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]2O)[C@@H](O)[C@@H]1O",
    "aspirin": "CC(=O)Oc1ccccc1C(=O)O",
}

---
## 2. Load pretrained models

Instantiate the pipeline — pretrained checkpoints are downloaded automatically on first use (~300 MB, cached afterwards).

In [ ]:
pipeline = NPClassifierPipeline()

print("Pipeline ready.")
print(f"  Pathway labels    ({len(pipeline._ensemble._pathway_labels)}): {pipeline._ensemble._pathway_labels}")
print(
    f"  Superclass labels ({len(pipeline._ensemble._superclass_labels)}): "
    f"first 5 = {pipeline._ensemble._superclass_labels[:5]}"
)
print(
    f"  Class labels      ({len(pipeline._ensemble._class_labels)}): first 5 = {pipeline._ensemble._class_labels[:5]}"
)

---
## 3. Full voted prediction — `.predict()`

Returns a dict (single SMILES) or list of dicts (batch).  
Keys: `"pathway"`, `"superclass"`, `"class"`, `"isglycoside"`.

In [ ]:
smi = MOLECULES["quercetin"]

result = pipeline.predict(smi)

print("Quercetin:")
print(f"  Pathway    : {result['pathway']}")
print(f"  Superclass : {result['superclass']}")
print(f"  Class      : {result['class']}")
print(f"  Glycoside  : {result['isglycoside']}")

In [ ]:
smiles_list = list(MOLECULES.values())
names_list = list(MOLECULES.keys())

results = pipeline.predict(smiles_list, check_glycoside=True)

rows = []
for name, res in zip(names_list, results):
    rows.append(
        {
            "molecule": name,
            "pathway": ", ".join(res["pathway"]) or "—",
            "superclass": ", ".join(res["superclass"]) or "—",
            "class": ", ".join(res["class"]) or "—",
            "glycoside": res["isglycoside"],
        }
    )

pd.DataFrame(rows).set_index("molecule")

---
## 4. Level-specific predictions

Convenience shortcuts that return just one hierarchy level.  
Single SMILES → list of strings. List of SMILES → list of lists.

In [ ]:
smi = MOLECULES["morphine"]

print("Morphine")
print("  .predict_pathway()   :", pipeline.predict_pathway(smi))
print("  .predict_superclass():", pipeline.predict_superclass(smi))
print("  .predict_class()     :", pipeline.predict_class(smi))

In [ ]:
class_preds = pipeline.predict_class(smiles_list)

for name, classes in zip(names_list, class_preds):
    print(f"  {name:<14}: {classes or ['—']}")

---
## 5. Molecular embeddings — `.predict_embeddings()`

Extracts the penultimate-layer activations from the chosen model (`"pathway"`, `"superclass"`, or `"class"`).  
Returns a `(N, 1536)` float32 array for the default architecture.

In [ ]:
embeddings = pipeline.predict_embeddings(smiles_list, level="class")

print(f"Embedding matrix: {embeddings.shape}   (N_molecules × embedding_dim)")
print(f"  dtype  : {embeddings.dtype}")
print(f"  range  : [{embeddings.min():.4f}, {embeddings.max():.4f}]")

### 5.1 Visualise with UMAP

Here we featurize the full training set, extract class-model embeddings, and project to 2-D.

In [ ]:
try:
    import umap
except ImportError:
    print("Install umap-learn to run this cell:  pip install umap-learn")
    raise

df_train = pd.read_csv(TEST_CSV)
label_cols = df_train.columns[LABEL_START:].tolist()

# Sample 500 molecules for speed
rng = np.random.default_rng(0)
idx = rng.choice(len(df_train), min(500, len(df_train)), replace=False)
df_sub = df_train.iloc[idx]
smiles = df_sub[SMILES_COL].tolist()

emb = pipeline.predict_embeddings(smiles, level="class", batch_size=64)

reducer = umap.UMAP(n_components=2, random_state=42)
coords = reducer.fit_transform(emb)

# Colour by the first pathway label that is active
pathway_labels_all = pipeline._label_names[:7]
pathway_matrix = df_sub[pathway_labels_all].values
colour_idx = np.argmax(pathway_matrix, axis=1)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(coords[:, 0], coords[:, 1], c=colour_idx, cmap="tab10", s=14, alpha=0.7)
cbar = plt.colorbar(scatter, ax=ax, ticks=range(len(pathway_labels_all)))
cbar.ax.set_yticklabels(pathway_labels_all, fontsize=7)
ax.set_title("UMAP of class-model embeddings (coloured by pathway)", fontsize=11)
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
plt.tight_layout()
plt.show()

---
## 6. Three-level SHAP explanation — `.explain()`

Runs ensemble voting, computes SHAP values for each predicted level with `shap.GradientExplainer`, and returns a combined matplotlib Figure:

- **Top row**: molecule with top-k bits highlighted + SHAP bar chart.
- **Bottom row**: grid of Morgan-environment fragments for those bits.

The `background` argument accepts a list of SMILES (featurized internally) **or** a pre-computed `(N, 6144)` feature array.  
A random sample of 50–200 training molecules gives a good balance of stability and speed.

In [ ]:
# Build a background set from training SMILES
rng = np.random.default_rng(42)
all_smiles = pd.read_csv(TRAIN_CSV)[SMILES_COL].dropna().tolist()
bg_smiles = [all_smiles[i] for i in rng.choice(len(all_smiles), 100, replace=False)]

print(f"Background: {len(bg_smiles)} molecules")

In [ ]:
# Full three-level explanation for quercetin
fig = pipeline.explain(
    MOLECULES["quercetin"],
    background=bg_smiles,  # or pass a pre-computed (N, 6144) array
    k=6,
)
plt.show()

In [ ]:
# Pre-compute background features once to avoid re-featurizing on every call
featurizer = pipeline.featurizer
bg_features = featurizer.transform(bg_smiles)  # (100, 6144)

for name, smi in MOLECULES.items():
    print(f"\n{'=' * 60}\n  {name.upper()}\n{'=' * 60}")
    fig = pipeline.explain(smi, background=bg_features, k=6)
    fig.suptitle(name.title(), fontsize=13, fontweight="bold", y=1.02)
    plt.show()

---
## 7. Level-specific SHAP bits — `.explain_bits()`

`.explain_bits()` focuses on a single hierarchy level and produces one explanation panel per predicted label at that level — useful when you want to drill into, say, only the class-level explanation.

In [ ]:
fig = pipeline.explain_bits(
    MOLECULES["quercetin"],
    level="class",
    background=bg_features,
    k=6,
)
plt.show()

In [ ]:
fig = pipeline.explain_bits(
    MOLECULES["morphine"],
    level="superclass",
    background=bg_features,
    k=6,
)
plt.show()

In [ ]:
fig = pipeline.explain_bits(
    MOLECULES["caffeine"],
    level="pathway",
    background=bg_features,
    k=6,
)
plt.show()

---
## 8. Save figures

All drawing methods return a `matplotlib.Figure` — save with `.savefig()`.

In [ ]:
import os

os.makedirs("pipeline_outputs", exist_ok=True)

for name, smi in MOLECULES.items():
    # Combined three-level figure
    fig = pipeline.explain(smi, background=bg_features, k=6)
    path = f"pipeline_outputs/{name}_explain.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Class-level fragment grid
    fig_cls = pipeline.explain_bits(smi, level="class", background=bg_features, k=6)
    path_cls = f"pipeline_outputs/{name}_class_bits.png"
    fig_cls.savefig(path_cls, dpi=150, bbox_inches="tight")
    plt.close(fig_cls)

    print(f"Saved {path}  +  {path_cls}")